In [2]:
import pandas as pd
import numpy as np




In [3]:
df=pd.read_csv('c:\\EcopackAI\\Dataset\\material.csv')
print(df)

     material_id                       material_name  \
0              1     Recycled cardboard / paperboard   
1              2  Molded pulp (recycled paper fiber)   
2              3    Kraft paper wraps, paper padding   
3              4          Starch-based foams/peanuts   
4              5     Mycelium-based molded packaging   
..           ...                                 ...   
106          107                   PHB bottles/trays   
107          108      Polyhydroxybutyrate composites   
108          109     Spider silk-inspired synthetics   
109          110                   Nettle fiber bags   
110          111              Willow bark composites   

                  material_type  strength_score  weight_capacity_score  \
0      Fiber-based (cellulosic)               3                      3   
1      Fiber-based (cellulosic)               3                      3   
2                   Fiber-based               2                      2   
3    Bio-based polymer (starch)

In [4]:
df.head()

,material_id,material_name,material_type,strength_score,weight_capacity_score,biodegradability_score,co2_emission_score,recyclability_percent
0,1,Recycled cardboard / paperboard,Fiber-based (cellulosic),3,3,85,70,75
1,2,Molded pulp (recycled paper fiber),Fiber-based (cellulosic),3,3,85,70,70
2,3,"Kraft paper wraps, paper padding",Fiber-based,2,2,90,70,70
3,4,Starch-based foams/peanuts,Bio-based polymer (starch),2,3,95,85,30
4,5,Mycelium-based molded packaging,Bio-based composite,3,3,95,90,20


In [5]:
#CO2 Impact Index
df['co2_impact_index'] = 1 / (df['co2_emission_score'] + 1)
#Normalized
df['co2_impact_index'] = (
    df['co2_impact_index'] - df['co2_impact_index'].min()
) / (
    df['co2_impact_index'].max() - df['co2_impact_index'].min()
)
print(df['co2_impact_index'])

0      0.399061
1      0.399061
2      0.399061
3      0.131783
4      0.062271
         ...   
106    0.209877
107    0.177510
108    0.089139
109    0.103030
110    0.131783
Name: co2_impact_index, Length: 111, dtype: float64


In [6]:
#Cost Efficiency Index
df['cost_efficiency_index'] = (
    df['strength_score'] + df['weight_capacity_score']
) / 2
#Normalized
df['cost_efficiency_index'] = (
    df['cost_efficiency_index'] - df['cost_efficiency_index'].min()
) / (
    df['cost_efficiency_index'].max() - df['cost_efficiency_index'].min()
)


In [7]:
#Material Suitability Score 
df['material_suitability_score'] = (
    0.25 * df['strength_score'] +
    0.25 * df['weight_capacity_score'] +
    0.20 * df['biodegradability_score'] +
    0.15 * (df['recyclability_percent'] / 100) +
    0.15 * df['co2_impact_index'])
#Normalized
df['material_suitability_score'] = (
 df['material_suitability_score'] - df['material_suitability_score'].min()
 ) / (
 df['material_suitability_score'].max() - df['material_suitability_score'].min()
 )


In [8]:
df.head()

,material_id,material_name,material_type,strength_score,weight_capacity_score,biodegradability_score,co2_emission_score,recyclability_percent,co2_impact_index,cost_efficiency_index,material_suitability_score
0,1,Recycled cardboard / paperboard,Fiber-based (cellulosic),3,3,85,70,75,0.399061,0.333333,0.875549
1,2,Molded pulp (recycled paper fiber),Fiber-based (cellulosic),3,3,85,70,70,0.399061,0.333333,0.875154
2,3,"Kraft paper wraps, paper padding",Fiber-based,2,2,90,70,70,0.399061,0.000000,0.901443
3,4,Starch-based foams/peanuts,Bio-based polymer (starch),2,3,95,85,30,0.131783,0.166667,0.961903
4,5,Mycelium-based molded packaging,Bio-based composite,3,3,95,90,20,0.062271,0.333333,0.973711


In [9]:
#feature matrix
X = df[[
 'strength_score',
 'weight_capacity_score',
 'biodegradability_score',
 'recyclability_percent',
'cost_efficiency_index']]  
print(X)

     strength_score  weight_capacity_score  biodegradability_score  \
0                 3                      3                      85   
1                 3                      3                      85   
2                 2                      2                      90   
3                 2                      3                      95   
4                 3                      3                      95   
..              ...                    ...                     ...   
106               4                      4                      90   
107               4                      4                      92   
108               5                      4                      85   
109               4                      4                      90   
110               3                      3                      88   

     recyclability_percent  cost_efficiency_index  
0                       75               0.333333  
1                       70               0.333333  
2  

In [10]:
y_cost = df['cost_efficiency_index']
print(y_cost)

0      0.333333
1      0.333333
2      0.000000
3      0.166667
4      0.333333
         ...   
106    0.666667
107    0.666667
108    0.833333
109    0.666667
110    0.333333
Name: cost_efficiency_index, Length: 111, dtype: float64


In [11]:
y_co2 = df['co2_impact_index'] 
print(y_co2)

0      0.399061
1      0.399061
2      0.399061
3      0.131783
4      0.062271
         ...   
106    0.209877
107    0.177510
108    0.089139
109    0.103030
110    0.131783
Name: co2_impact_index, Length: 111, dtype: float64


In [12]:
print(X.shape) 
print(y_cost.shape) 
print(y_co2.shape) 

(111, 5)
(111,)
(111,)


In [13]:
#spliting dataset
from sklearn.model_selection import train_test_split
X_train, X_test, y_cost_train, y_cost_test = train_test_split(
    X,
    y_cost,
    test_size=0.2,
    random_state=42
) 

In [14]:
print(X_train.shape) 
print(X_test.shape) 

(88, 5)
(23, 5)


In [15]:
#Cost prediction model (random forest)
from sklearn.ensemble import RandomForestRegressor
cost_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
cost_model.fit(X_train, y_cost_train) 


RandomForestRegressor(random_state=42)

In [17]:
#Cost model evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
cost_predictions = cost_model.predict(X_test)
print("MAE:", mean_absolute_error(y_cost_test, cost_predictions))
print("RMSE:", mean_squared_error(y_cost_test, cost_predictions, squared=False))
print("R2:", r2_score(y_cost_test, cost_predictions)) 

MAE: 0.004492753623188678
RMSE: 0.020863022323793154
R2: 0.9937768768768769


C:\Users\Aakansha\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [18]:
#Co2 impact model      (install XG Boost)
from xgboost import XGBRegressor
co2_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
co2_model.fit(X_train, y_co2.loc[y_cost_train.index])


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [19]:
#Co2 model evaluation (RMSE , R2 score)
co2_predictions = co2_model.predict(X_test)
print("MAE:", mean_absolute_error(y_co2.loc[y_cost_test.index], co2_predictions))
print("RMSE:", mean_squared_error(y_co2.loc[y_cost_test.index], co2_predictions, squared=False))
print("R2:", r2_score(y_co2.loc[y_cost_test.index], co2_predictions))


MAE: 0.048055026385834955
RMSE: 0.08600612416309206
R2: 0.8909842269425886


C:\Users\Aakansha\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [20]:
#prediction to ranking
df_test = df.loc[X_test.index].copy()
df_test['predicted_cost'] = cost_predictions
df_test['predicted_co2'] = co2_predictions 

In [21]:
#final ranking score 
df_test['final_score'] = (
    0.5 * df_test['predicted_cost'].rank(ascending=True) +
    0.5 * df_test['predicted_co2'].rank(ascending=True)
) 

In [22]:
#Output (core intellligence)
df_test.sort_values('final_score').head() 

,material_id,material_name,material_type,strength_score,weight_capacity_score,biodegradability_score,co2_emission_score,recyclability_percent,co2_impact_index,cost_efficiency_index,material_suitability_score,predicted_cost,predicted_co2,final_score
78,79,Pullulan films (fungal),Bio-based polysaccharide,2,2,95,92,25,0.036559,0.0,0.947613,0.0,0.080563,3.25
64,65,Sago palm starch films,Bio-based starch,2,2,95,80,25,0.209877,0.0,0.948980,0.0,0.080563,3.25
79,80,Xanthan gum coatings,Bio-based polysaccharide,2,2,90,88,30,0.089139,0.0,0.895844,0.0,0.085390,4.25
65,66,Jackfruit seed bioplastics,Bio-based composite,2,2,90,88,30,0.089139,0.0,0.895844,0.0,0.085390,4.25
68,69,Papaya leaf fibers,Agro-waste fiber,2,2,90,85,50,0.131783,0.0,0.897758,0.0,0.091664,5.00


In [23]:
#freeze model
import joblib
joblib.dump(cost_model, 'cost_model.pkl')
joblib.dump(co2_model, 'co2_model.pkl') 

['co2_model.pkl']